# Phase 1: Exploratory Data Analysis (EDA)
## COVID-19 Cough Detection Dataset

This notebook provides comprehensive exploratory analysis of the CoughVID dataset including:
- Dataset overview and statistics
- Data quality assessment
- Demographic distributions
- Data missingness analysis
- Statistical summaries

In [ ]:
import json
import os
import sys
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully")

In [ ]:
def load_metadata(dataset_dir):
    """Load all metadata JSON files from dataset directory"""
    metadata_rows = []
    dataset_path = Path(dataset_dir)
    
    if not dataset_path.is_dir():
        raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")
    
    json_files = sorted(dataset_path.glob("*.json"))
    logger.info(f"Found {len(json_files)} JSON metadata files")
    
    for file_path in json_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                metadata = json.load(f)
            metadata['file_id'] = file_path.stem
            metadata_rows.append(metadata)
        except Exception as e:
            logger.warning(f"Error loading {file_path.name}: {e}")
    
    return pd.DataFrame(metadata_rows)

# Load dataset
dataset_dir = "public_dataset"
df = load_metadata(dataset_dir)
print(f"\n✓ Loaded {len(df)} records from {dataset_dir}")
print(f"✓ DataFrame shape: {df.shape}")

In [ ]:
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\nDataset Shape: {df.shape}")
print(f"Total Records: {len(df)}")
print(f"Total Features: {len(df.columns)}")
print("\nColumn Names and Types:")
print(df.dtypes)
print("\nFirst Few Records:")
df.head()

In [ ]:
print("=" * 80)
print("DATA QUALITY & MISSINGNESS ANALYSIS")
print("=" * 80)

# Missingness analysis
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_data.values,
    'Missing_Percent': missing_percent.values
}).sort_values('Missing_Count', ascending=False)

print("\nMissing Data Summary:")
print(missing_df.to_string())

# Visualize missingness
fig, ax = plt.subplots(figsize=(12, 6))
missing_df[missing_df['Missing_Count'] > 0].plot(
    x='Column', y='Missing_Percent', kind='barh', ax=ax, color='coral'
)
ax.set_xlabel('Percentage Missing (%)')
ax.set_title('Data Missingness Analysis')
plt.tight_layout()
plt.savefig('eda_missingness.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Missingness plot saved as 'eda_missingness.png'")

In [ ]:
print("=" * 80)
print("STATUS/LABEL DISTRIBUTION")
print("=" * 80)

status_dist = df['status'].value_counts()
print("\nStatus Distribution:")
print(status_dist)
print(f"\nProportions:")
print(df['status'].value_counts(normalize=True))

# Visualize status distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
status_dist.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral', 'lightgreen'])
axes[0].set_title('Status Distribution (Count)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Status')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
status_dist.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral', 'lightgreen'])
axes[1].set_title('Status Distribution (Proportion)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('eda_status_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Status distribution plot saved")

In [ ]:
print("=" * 80)
print("DEMOGRAPHIC ANALYSIS")
print("=" * 80)

# Gender distribution
if 'gender' in df.columns:
    print("\nGender Distribution:")
    gender_dist = df['gender'].value_counts()
    print(gender_dist)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gender overall
    gender_dist.plot(kind='bar', ax=axes[0], color=['lightblue', 'lightpink'])
    axes[0].set_title('Gender Distribution', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)
    
    # Gender by status
    gender_status = pd.crosstab(df['gender'], df['status'])
    gender_status.plot(kind='bar', ax=axes[1])
    axes[1].set_title('Gender Distribution by Health Status', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Count')
    axes[1].set_xlabel('Gender')
    axes[1].legend(title='Status')
    axes[1].tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.savefig('eda_gender_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Gender analysis plot saved")

In [ ]:
print("=" * 80)
print("NUMERIC FEATURES ANALYSIS")
print("=" * 80)

# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric Features: {numeric_cols}")

if numeric_cols:
    stats_summary = df[numeric_cols].describe()
    print("\nNumeric Features Statistics:")
    print(stats_summary)
    
    # Save stats to CSV
    stats_summary.to_csv('eda_numeric_statistics.csv')
    print("\n✓ Statistics saved to 'eda_numeric_statistics.csv'")
    
    # Visualize distributions
    fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(12, 4*len(numeric_cols)))
    if len(numeric_cols) == 1:
        axes = [axes]
    
    for idx, col in enumerate(numeric_cols):
        axes[idx].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.savefig('eda_numeric_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Numeric distributions plot saved")

In [ ]:
print("=" * 80)
print("CORRELATION & FEATURE RELATIONSHIPS")
print("=" * 80)

if len(numeric_cols) > 1:
    # Correlation matrix
    corr_matrix = df[numeric_cols].corr()
    print("\nCorrelation Matrix:")
    print(corr_matrix)
    
    # Visualize correlation
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                square=True, ax=ax, cbar_kws={'label': 'Correlation'})
    ax.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('eda_correlation_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✓ Correlation matrix plot saved")
    
    # Save correlation matrix
    corr_matrix.to_csv('eda_correlation_matrix.csv')
    print("✓ Correlation matrix saved to CSV")

In [ ]:
print("=" * 80)
print("SUMMARY STATISTICS BY STATUS")
print("=" * 80)

# Group statistics by status
print("\nStatistics by Health Status:")
for status in df['status'].unique():
    status_data = df[df['status'] == status]
    print(f"\n{'='*40}")
    print(f"Status: {status}")
    print(f"{'='*40}")
    print(f"Count: {len(status_data)}")
    if 'gender' in status_data.columns:
        print(f"\nGender Distribution:")
        print(status_data['gender'].value_counts())
    
    if numeric_cols:
        print(f"\nNumeric Features Summary:")
        print(status_data[numeric_cols].describe().round(3))

# Save comprehensive summary
summary_by_status = df.groupby('status').describe()
summary_by_status.to_csv('eda_summary_by_status.csv')
print("\n✓ Summary by status saved to CSV")

In [ ]:
print("=" * 80)
print("DATA QUALITY SUMMARY REPORT")
print("=" * 80)

report = {
    'Total Records': len(df),
    'Total Features': len(df.columns),
    'Numeric Features': len(numeric_cols),
    'Categorical Features': len(df.columns) - len(numeric_cols),
    'Memory Usage (MB)': df.memory_usage(deep=True).sum() / 1024**2,
    'Duplicate Rows': df.map(lambda value: json.dumps(value, sort_keys=True) if isinstance(value, (dict, list)) else value).duplicated().sum(),
    'Complete Cases': len(df.dropna()),
    'Records with Missing Values': df.isnull().any(axis=1).sum(),
}

print("\nData Quality Metrics:")
for key, value in report.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

# Save report
import json
with open('eda_quality_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=lambda value: value.item())
print("\n✓ Quality report saved to 'eda_quality_report.json'")

In [1]:
print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print("""
✓ Phase 1 EDA Complete!

Generated Outputs:
  1. eda_missingness.png - Missingness analysis visualization
  2. eda_status_distribution.png - Status label distribution
  3. eda_gender_analysis.png - Gender demographic analysis  
  4. eda_numeric_distributions.png - Numeric feature distributions
  5. eda_correlation_matrix.png - Feature correlation heatmap
  6. eda_numeric_statistics.csv - Numeric statistics
  7. eda_correlation_matrix.csv - Correlation values
  8. eda_summary_by_status.csv - Statistics grouped by status
  9. eda_quality_report.json - Data quality metrics

Key Findings:
  - Dataset size: {} records
  - Classes: {}
  - Missing data: {}%
  - Feature completeness: {:.1f}%
  
Ready for Phase 2: Feature Extraction
""".format(
    len(df),
    df['status'].nunique(),
    round((df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100, 2),
    ((df.shape[0] * df.shape[1] - df.isnull().sum().sum()) / (df.shape[0] * df.shape[1])) * 100
))

FINAL SUMMARY


NameError: name 'df' is not defined